# script_04_annotate — Anotación pragmática y sentimental con Gemini y Antigravity SDK

Este cuaderno implementa la anotación lingüística automatizada de un corpus de diálogos japonés-español utilizando **Google Antigravity SDK** y el modelo **Gemini 3.5 Flash** con salidas estructuradas nativas de Pydantic.

In [ ]:
import asyncio
import json
import os
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional, Literal

import pydantic
from google.antigravity import Agent, LocalAgentConfig

# ── Configuración ─────────────────────────────────────────────────────────────

SAMPLE_PATH       = Path("data/processed/dataset_sample.jsonl")
PILOT_SAMPLE_PATH = Path("data/processed/pilot_sample.json")
PILOT_ANNOT_PATH  = Path("data/processed/pilot_annotations.json")
ANNOTATED_PATH    = Path("data/processed/corpus_annotated.json")
STATS_PATH        = Path("data/processed/annotation_stats.json")

# Clave de API de Gemini cargada desde variable de entorno
GEMINI_API_KEY    = os.environ["GEMINI_API_KEY"]
MODEL_NAME        = "gemini-3.5-flash"

PILOT_PER_PARTICLE = 20
MAX_RETRIES        = 3
RETRY_BASE_DELAY   = 2.0         # segundos
PROGRESS_EVERY     = 50
RANDOM_SEED        = 42

In [ ]:
# Definición del esquema con Pydantic para la salida estructurada de Antigravity
class AnnotationResponse(pydantic.BaseModel):
    document_id: str
    context_es_corrected: str
    current_es_corrected: str
    intent: Literal["Afirmación", "Rechazo", "Solicitud", "Pregunta", "Duda"]
    sentiment: Literal["Positivo", "Negativo", "Neutral", "Incierto"]

In [ ]:
ANNOTATION_PROMPT = """Eres un anotador lingüístico especializado en pragmática japonesa, análisis discursivo y traducción japonés-español.

OBJETIVO

Realizar una anotación piloto de un corpus japonés-español para validar una taxonomía de intención comunicativa y tono afectivo-pragmático.

La anotación será posteriormente revisada manualmente sobre una muestra representativa de aproximadamente 20 ejemplos por partícula para evaluar la consistencia del etiquetado.

Tu función es asistir el proceso de anotación, no reinterpretar libremente los datos.

TAREAS

Para cada registro:

1. Analizar conjuntamente:
   * context_ja
   * current_ja
   * context_es
   * current_es
   * particle
   * particles_detected

2. Revisar las traducciones de:
   * context_es
   * current_es

3. Corregir únicamente cuando exista un error real de traducción, omisión relevante, adición de significado inexistente o pérdida pragmática importante respecto al japonés original.

4. Etiquetar:
   * intent
   * sentiment

REGLAS DE TRADUCCIÓN

* NO reformules por estilo.
* NO sustituyas expresiones simplemente porque existe una traducción mejor.
* NO modernice el lenguaje.
* NO hagas cambios cosméticos.
* Conserva la traducción original cuando sea aceptable y transmita correctamente el significado.

IMPORTANTE:

Si la traducción es adecuada, devuelve exactamente la misma traducción.
La corrección debe ser conservadora.
Sólo modifica una traducción cuando exista evidencia clara de que no representa correctamente el contenido japonés.

REGLAS DE ANOTACIÓN

* Analiza principalmente current_ja junto con context_ja.
* Utiliza context_es y current_es únicamente como apoyo interpretativo.
* La partícula es una evidencia secundaria.
* No asumas que una partícula determina automáticamente una categoría.
* Si existe conflicto entre la partícula y el significado global del enunciado, prevalece el significado global.
* Selecciona exactamente una categoría de intención.
* Selecciona exactamente una categoría de sentimiento.
* Nunca inventes categorías nuevas.
* No expliques tu razonamiento.
* No agregues comentarios.
* No agregues campos adicionales.

TAXONOMÍA DE INTENCIÓN

Afirmación: Acuerdo, aceptación o confirmación genuina (Marcadores frecuentes: はい, そうです, そうだ, よ)
Rechazo: Negación, desacuerdo, oposición o rechazo directo o indirecto (Marcadores frecuentes: いいえ, de mo, けど, けれど, けれども)
Solicitud: Petición o requerimiento dirigido al interlocutor (Marcadores frecuentes: ～てください, ～ていただけますか, ～てもらえますか)
Pregunta: Búsqueda de información o confirmación (Marcadores frecuentes: ～ですか, ～でしょうか, か final)
Duda: Incertidumbre, vacilación, reflexión o falta de seguridad (Marcadores frecuentes: ～kana, ～daroaka, んだけど)

TAXONOMÍA DE TONO AFECTIVO-PRAGMÁTICO

Positivo: Tono cálido, amistoso, cooperativo o de aprobación.
Negativo: Tono agresivo, tenso, amenazante, frustrado o de desaprobación.
Neutral: Tono descriptivo o informativo sin carga afectiva marcada.
Incierto: Tono dubitativo, vacilante, reflexivo o especulativo.

SALIDA

Devuelve únicamente un objeto JSON válido con la siguiente estructura:

{\n\"document_id\": \"...\",\n\"context_es_corrected\": \"...\",\n\"current_es_corrected\": \"...\",\n\"intent\": \"Afirmación|Rechazo|Solicitud|Pregunta|Duda\",\n\"sentiment\": \"Positivo|Negativo|Neutral|Incierto\"\n}

Si una traducción no requiere cambios:
\"context_es_corrected\" debe ser idéntico a \"context_es\".
\"current_es_corrected\" debe ser idéntico a \"current_es\".

No incluyas explicaciones.
No uses Markdown.
No envuelvas la respuesta en bloques de código.

REGISTRO A PROCESAR:
{{INPUT_JSON}}"""

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_json(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
async def call_gemini(record: dict, agent: Agent) -> Optional[dict]:
    prompt = ANNOTATION_PROMPT.replace(
        "{{INPUT_JSON}}",
        json.dumps(record, ensure_ascii=False, indent=2)
    )
    for attempt in range(MAX_RETRIES):
        try:
            response = await agent.chat(prompt)
            result = await response.structured_output()
            if result:
                if not isinstance(result, dict):
                    result = result.model_dump()
                return result
            print(f"    ✗ respuesta inválida o falló validación del esquema (intento {attempt + 1})")
        except Exception as e:
            print(f"    ✗ error API / procesar (intento {attempt + 1}): {e}")
        if attempt < MAX_RETRIES - 1:
            delay = RETRY_BASE_DELAY * (2 ** attempt)
            await asyncio.sleep(delay)
    return None

In [ ]:
def sample_stratified(records: list[dict], per_particle: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    by_particle: dict[str, list] = defaultdict(list)
    for rec in records:
        p = rec.get("particle")
        if p:
            by_particle[p].append(rec)
    sample = []
    for p in sorted(by_particle):
        group = by_particle[p]
        n = min(per_particle, len(group))
        chosen = rng.sample(group, n)
        sample.extend(chosen)
        if len(group) < per_particle:
            print(f"  ⚠  {p}: solo {len(group)} disponibles (solicitados {per_particle})")
    return sample

def compute_stats(annotations: list[dict]) -> dict:
    return {
        "total": len(annotations),
        "by_particle":  dict(sorted(Counter(a.get("particle")  for a in annotations).items())),
        "by_intent":    dict(sorted(Counter(a.get("intent")    for a in annotations).items())),
        "by_sentiment": dict(sorted(Counter(a.get("sentiment") for a in annotations).items())),
        "corrections": {
            "context": sum(
                1 for a in annotations
                if a.get("context_es_corrected") != a.get("context_es")
            ),
            "current": sum(
                1 for a in annotations
                if a.get("current_es_corrected") != a.get("current_es")
            ),
        },
    }

def print_stats(stats: dict):
    sep = "─" * 44
    print(f"\n  {sep}")
    print(f"  Total anotados : {stats['total']:,}")
    print(f"\n  Intent:")
    for k, v in stats["by_intent"].items():
        pct = 100 * v / max(stats["total"], 1)
        bar = "█" * max(1, int(pct / 3))
        print(f"    {k:<14} {v:>5,}  {pct:5.1f}%  {bar}")
    print(f"\n  Sentiment:")
    for k, v in stats["by_sentiment"].items():
        pct = 100 * v / max(stats["total"], 1)
        bar = "█" * max(1, int(pct / 3))
        print(f"    {k:<14} {v:>5,}  {pct:5.1f}%  {bar}")
    print(f"\n  Correcciones context_es : {stats['corrections']['context']}")
    print(f"  Correcciones current_es : {stats['corrections']['current']}")
    print(f"  {sep}")

In [ ]:
async def run_pilot(agent: Agent):
    print(f"\n{'═' * 50}")
    print("  FASE 1 — VALIDACIÓN PILOTO")
    print(f"{'═' * 50}\n")
    records = load_jsonl(SAMPLE_PATH)
    print(f"  Corpus cargado: {len(records):,} registros")
    pilot = sample_stratified(records, PILOT_PER_PARTICLE, RANDOM_SEED)
    save_json(pilot, PILOT_SAMPLE_PATH)
    print(f"  Muestra piloto: {len(pilot)} registros → {PILOT_SAMPLE_PATH}\n")
    annotations, errors = [], []
    for i, rec in enumerate(pilot, 1):
        result = await call_gemini(rec, agent)
        if result:
            merged = {**rec, **result}
            annotations.append(merged)
        else:
            errors.append({"index": i, "document_id": rec.get("document_id")})
        if i % PROGRESS_EVERY == 0 or i == len(pilot):
            print(f"  [{i:>4}/{len(pilot)}]  ok={len(annotations)}  errores={len(errors)}")
    save_json(annotations, PILOT_ANNOT_PATH)
    print(f"\n  Guardado: {PILOT_ANNOT_PATH}")
    if errors:
        print(f"  ✗ {len(errors)} registros fallidos:")
        for e in errors[:5]:
            print(f"    • [{e['index']}] {e['document_id']}")
        if len(errors) > 5:
            print(f"    … y {len(errors) - 5} más")
    stats = compute_stats(annotations)
    print_stats(stats)
    return {"annotated": len(annotations), "errors": errors, "stats": stats}

async def run_full(agent: Agent):
    print(f"\n{'═' * 50}")
    print("  FASE 2 — ANOTACIÓN COMPLETA")
    print(f"{'═' * 50}\n")
    records = load_jsonl(SAMPLE_PATH)
    total = len(records)
    print(f"  Total registros: {total:,}\n")
    annotations, errors = [], []
    for i, rec in enumerate(records, 1):
        result = await call_gemini(rec, agent)
        if result:
            annotations.append({**rec, **result})
        else:
            errors.append({"index": i, "document_id": rec.get("document_id")})
        if i % PROGRESS_EVERY == 0 or i == total:
            print(f"  [{i:>5}/{total}]  ok={len(annotations):,}  errores={len(errors):,}")
    save_json(annotations, ANNOTATED_PATH)
    print(f"\n  Guardado: {ANNOTATED_PATH}")
    stats = compute_stats(annotations)
    save_json(stats, STATS_PATH)
    print(f"  Stats:    {STATS_PATH}")
    if errors:
        print(f"\n  ✗ {len(errors)} registros fallidos (ver corpus_annotated.json para IDs faltantes)")
    print_stats(stats)
    return {"total": total, "annotated": len(annotations), "errors": len(errors), "stats": stats}

In [ ]:
# Inicializar configuración del Agente de Antigravity
config = LocalAgentConfig(
    model=MODEL_NAME,
    response_schema=AnnotationResponse,
    api_key=GEMINI_API_KEY
)

# Ejecutar validación piloto de manera asíncrona en el notebook
async with Agent(config=config) as agent:
    await run_pilot(agent)

In [ ]:
# Ejecutar anotación completa del corpus
# async with Agent(config=config) as agent:
#     await run_full(agent)